In [10]:
import pandas as pd
import re
from tqdm import tqdm

def extract_answer_final_v5(response_text: str) -> str:
    """
    [v5] 모델의 모든 답변 형식에서 정답(A, B, C, D)을 추출하는 가장 강력한 함수.
    """
    if not isinstance(response_text, str):
        return None
        
    text = response_text.strip()
    
    # 패턴 리스트 (신뢰도 순)
    patterns = [
        r'(?:answer|choice|option)\s*(?:is|:)?\s*\b([A-D])\b', # 1. "Answer is A", "option C"
        r'^\s*\(?([A-D])\)?[\.)]',                            # 2. 답변이 "A)" 등으로 시작
        r'^\s*([A-D])\s*$',                                   # 3. 답변 전체가 오직 A, B, C, D 중 하나
        r'\b([A-D])[\.)]'                                     # 4. 문장 중간의 "B)"
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).upper()

    return None

def keyword_fallback_guesser(caption: str, choices: dict) -> str:
    """
    모델 답변이 완전히 실패했을 때, 캡션과 선택지 간의 단순 키워드 매칭으로 정답을 추측합니다.
    """
    if not isinstance(caption, str):
        return 'A' 

    caption_words = set(re.findall(r'\w+', caption.lower()))
    
    best_match_count = -1
    best_option = 'A'

    for option, text in choices.items():
        if not isinstance(text, str):
            continue
        
        option_words = set(re.findall(r'\w+', text.lower()))
        overlap_count = len(caption_words.intersection(option_words))
        
        if overlap_count > best_match_count:
            best_match_count = overlap_count
            best_option = option
            
    return best_option


# --- 파일 경로 설정 ---
# 원본 데이터 파일 경로 (캡션, 질문, 선택지 포함)
original_data_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/finetuning/unilm/beit3/output_coco_captioning_inference/test_with_caption.csv' 
# 모델의 원본 출력이 저장된 파일 경로
model_output_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/llm/model_output.csv'
# 최종 제출 파일 이름
final_submission_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/llm/final_submission_6.csv'

print("Step 1: 데이터 파일들을 로드합니다...")
try:
    # ✨ 수정된 부분: 원본 데이터와 모델 출력 데이터를 모두 로드합니다.
    original_df = pd.read_csv(original_data_path)
    output_df = pd.read_csv(model_output_path)
except FileNotFoundError as e:
    print(f"오류: 파일을 찾을 수 없습니다 - {e}")
    # return

# ✨ 수정된 부분: 'ID'를 기준으로 두 데이터프레임을 병합합니다.
# 이제 df는 caption, A, B, C, D, model_output 등 모든 열을 가집니다.
df = pd.merge(original_df, output_df[['ID', 'model_output']], on='ID', how='left')

print(f"총 {len(df)}개의 데이터를 처리합니다.")

final_answers = []
answer_sources = [] 

print("\nStep 2: 정답 추출, 검증 및 대체 계획을 실행합니다...")
for _, row in tqdm(df.iterrows(), total=df.shape[0]):
    # 1단계: 모델 답변에서 정답 추출
    answer = extract_answer_final_v5(row['model_output'])
    
    if answer:
        final_answers.append(answer)
        answer_sources.append('Model')
    else:
        # 2&3단계: 추출 실패 시, 키워드 기반 대체 계획 실행
        # ✨ 이제 row에는 'A', 'B', 'C', 'D' 열이 안전하게 존재합니다.
        choices_dict = {'A': row['A'], 'B': row['B'], 'C': row['C'], 'D': row['D']}
        fallback_answer = keyword_fallback_guesser(row['caption'], choices_dict)
        final_answers.append(fallback_answer)
        answer_sources.append('Fallback')

# 최종 제출 DataFrame 생성
submission_df = pd.DataFrame({
    'ID': df['ID'],
    'answer': final_answers,
    'source': answer_sources
})

print(f"\nStep 3: 최종 결과를 '{final_submission_path}' 파일로 저장합니다...")
submission_df[['ID', 'answer']].to_csv(final_submission_path, index=False)

print("완료되었습니다.")
print("\n처리 결과 요약:")
print(submission_df['source'].value_counts())
print("\n최종 제출 파일 내용 (상위 5개):")
print(submission_df.head())

Step 1: 데이터 파일들을 로드합니다...
총 852개의 데이터를 처리합니다.

Step 2: 정답 추출, 검증 및 대체 계획을 실행합니다...


100%|██████████| 852/852 [00:00<00:00, 23011.35it/s]


Step 3: 최종 결과를 '/data/2_data_server/cv-07/dice/2025_samsung_challenge/llm/final_submission_6.csv' 파일로 저장합니다...
완료되었습니다.

처리 결과 요약:
source
Model       847
Fallback      5
Name: count, dtype: int64

최종 제출 파일 내용 (상위 5개):
         ID answer source
0  TEST_000      B  Model
1  TEST_001      A  Model
2  TEST_002      B  Model
3  TEST_003      B  Model
4  TEST_004      A  Model
